# Session 3: When the model learns the answers by heart

**259.075 Data-integrated Algorithmic Design Processes II** · Session 3 of 7

Last week ended with a network of sixteen kinks following a curve closely, and with a question:
what happens if you keep adding kinks until the model passes through every single point?

The answer is the most useful thing in this course, and it comes in two parts, because there are
two different ways for a model to be confidently wrong. One is overfitting, which you have
probably heard of. The other is extrapolation, which is less talked about and, on the numbers
below, considerably worse.

You also write the training loop yourself today, in PyTorch, three lines of it.

## Noise, and why it is there on purpose

Until now the target came out of a formula, and any sufficiently large model could hit it
exactly. From here on there is noise on it.

The reason is dull. Vienna publishes no open dataset of measured consumption per building, and
what consumption figures exist are aggregated. So instead of pretending, we take the same
invented curve as last week and add a random amount to each building, drawn from a normal
distribution with a standard deviation of 20 kWh/m²a. That number is not a measurement either.
It stands in for everything a formula cannot know about a house: who lives there, whether the
windows are open in February, what the neighbours do.

> Noise is not dirt in the data. It is everything the model cannot possibly predict, and the
> whole trick is not to try.

The training set is deliberately tiny, twenty-five buildings. Overfitting is a relationship
between how much model you have and how much data, and with twenty-five points you can get on
the wrong side of that relationship inside two hours.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

zufall = np.random.default_rng(7)
df = pd.read_csv("../data/gebaeude_wien.csv")

k = zufall.choice(df["kompaktheit"].to_numpy(), size=265, replace=False)
rein = 40 + 9 * k + 45 * np.sin(0.8 * k)          # the same rule as last week
y = rein + zufall.normal(0, 20, size=265)         # plus what no rule can know

i = np.arange(len(k))
zufall.shuffle(i)
i_tr, i_va, i_te = i[:25], i[25:105], i[105:]
print(f"train {len(i_tr)}, validation {len(i_va)}, test {len(i_te)}")

The split happens before anything is looked at, including the first plot. Everything from here
to the end of the session uses the training and validation buildings only; the test buildings
are not drawn, not summarised, and not used for axis limits.

That is stricter than it needs to be for the fitted parameters, and it is the habit worth
having: the moment you have seen the test set, you cannot un-see it when choosing the next
thing to try.

In [ ]:
sichtbar = np.concatenate([i_tr, i_va])
ordnung = i_tr[np.argsort(k[i_tr])]

fig = go.Figure()
fig.add_trace(go.Scatter(x=k[ordnung], y=rein[ordnung], mode="lines", name="the rule",
                         line=dict(width=3, color="#4c78a8")))
fig.add_trace(go.Scatter(x=k[i_va], y=y[i_va], mode="markers", name="validation",
                         marker=dict(size=5, color="#8b95a8", opacity=0.6)))
fig.add_trace(go.Scatter(x=k[i_tr], y=y[i_tr], mode="markers", name="training",
                         marker=dict(size=9, color="crimson")))
fig.update_layout(height=430, xaxis_title="compactness", yaxis_title="target",
                  title="the curve underneath, and what the model gets to see")
fig.show()

## Split first, then scale

The *training* set is what the model learns from. The *validation* set is what you look at while
deciding anything: how big the model should be, when to stop, which learning rate. The *test*
set you open once, at the very end, and then you are finished with it.

The trap is in the scaling. Mean and standard deviation have to come from the training part
alone. Compute them over all three and you have let the model peek at buildings it is not
supposed to have seen, and your test number is worth less than you think. The mistake has a
friendly name, *leakage*, and it is the most common way a published result turns out to be
nonsense.

> Anything computed from the data is part of the model. That includes the mean you subtracted.

Note the range while you are here: the twenty-five training buildings cover compactness from
about 4.0 to 10.0, while the validation set reaches 11.4. Nothing about
that is unusual, and it matters a great deal in two cells' time.

In [ ]:
k_m, k_s = k[i_tr].mean(), k[i_tr].std()          # from the training part only
y_m, y_s = y[i_tr].mean(), y[i_tr].std()

k_lo, k_hi = k[i_tr].min(), k[i_tr].max()
print(f"scaling taken from the training part: mean {k_m:.3f}, spread {k_s:.3f}")
print(f"training range of compactness: {k_lo:.2f} to {k_hi:.2f}")
print(f"validation reaches {k[i_va].max():.2f}, "
      f"{int((k[i_va] > k_hi).sum())} buildings lie outside the training range")

## Capacity, on a dial

Before the network, a model whose capacity sits in a single number: a polynomial. Degree one is
a straight line, degree three can bend twice, degree nine can bend eight times. Fitting one is
least squares from session 1, nothing new.

Four of them on the same twenty-five buildings. The dashed lines mark the training range; the
degree-nine curve leaves the picture just outside it and does not come back.

In [ ]:
def polynom(grad):
    """Least squares on scaled compactness; scaling keeps the high powers well behaved."""
    c = np.polyfit((k[i_tr] - k_m) / k_s, y[i_tr], grad)

    def vor(werte):
        return np.polyval(c, (werte - k_m) / k_s)

    drinnen = (k[i_va] >= k_lo) & (k[i_va] <= k_hi)
    return dict(vor=vor,
                train=float(np.mean((vor(k[i_tr]) - y[i_tr]) ** 2)),
                val=float(np.mean((vor(k[i_va]) - y[i_va]) ** 2)),
                val_innen=float(np.mean((vor(k[i_va][drinnen]) - y[i_va][drinnen]) ** 2)),
                val_aussen=float(np.mean((vor(k[i_va][~drinnen]) - y[i_va][~drinnen]) ** 2)))


grade = [1, 3, 6, 9]
gitter_k = np.linspace(k[sichtbar].min(), k[sichtbar].max(), 400)

fig = make_subplots(rows=2, cols=2, subplot_titles=[f"degree {g}" for g in grade])
for n, grad in enumerate(grade):
    e = polynom(grad)
    r, c = divmod(n, 2)
    fig.add_trace(go.Scatter(x=k[i_va], y=y[i_va], mode="markers", showlegend=False,
                             marker=dict(size=4, color="#8b95a8", opacity=0.5)),
                  row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=k[i_tr], y=y[i_tr], mode="markers", showlegend=False,
                             marker=dict(size=7, color="crimson")), row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=gitter_k, y=e["vor"](gitter_k), mode="lines", showlegend=False,
                             line=dict(width=3, color="#d8a24a")), row=r + 1, col=c + 1)
    for grenze in (k_lo, k_hi):
        fig.add_vline(x=grenze, line_dash="dot", line_color="#556", row=r + 1, col=c + 1)
    fig.layout.annotations[n].text = (f"degree {grad} — train {e['train']:.0f}, "
                                      f"validation inside {e['val_innen']:,.0f}")
fig.update_layout(height=660, title="25 training buildings in red, 80 validation buildings in "
                                    "grey, training range between the dotted lines")
fig.update_yaxes(range=[y[sichtbar].min() - 20, y[sichtbar].max() + 20])
fig.update_xaxes(range=[k[sichtbar].min() - 0.2, k[sichtbar].max() + 0.2])
fig.show()

In [ ]:
print(f"{'degree':>6} {'train':>10} {'val inside':>12} {'val outside':>18}")
for grad in [1, 3, 5, 6, 9, 12]:
    e = polynom(grad)
    print(f"{grad:>6} {e['train']:>10.0f} {e['val_innen']:>12.0f} {e['val_aussen']:>18,.0f}")
print(f"\ninside = the 78 validation buildings within the training range")
print(f"outside = the 2 beyond it")

Two separate things are going wrong, and the table separates them.

Inside the training range, the error falls to a minimum around degree five and then climbs
again: from about 500 at degree five to over a thousand at degree nine. That is overfitting. The
curve is spending its flexibility on the noise in twenty-five particular buildings, and on other
buildings drawn the same way it does worse. Notice that the climb is not tidy; the numbers wobble
from degree to degree, because each degree refits everything from scratch on a small sample.

Outside the training range, at the two buildings with compactness above ten, degree nine has a
mean squared error of over two billion: a typical miss of about 48,000 on values around a
hundred. The curve does not gently get worse out there, it leaves the page. A
polynomial has no opinion about territory it has not seen, and its extrapolation is whatever the
highest power happens to do.

> Overfitting costs you a factor. Extrapolation costs you everything, and nothing in the output
> warns you which one you are doing.

The training error, meanwhile, falls with every degree and never turns around. Twenty-five
points and degree nine is not yet enough to pass through them all, but it is enough to chase
them, and chasing is what the training error rewards.

> The training error is not a measure of quality. It is a measure of how hard the model tried.

## The same thing, with a network

A network has no single dial. Its capacity sits in the width, in the number of layers, and in
the activation function, all of which you fix before training starts. How long you train does
not change what the network could represent, but it does change which function you end up with,
and stopping early is one of the ways to get a smoother one.

So the loop from last week comes back, and this time you write it. Three lines are missing, and
they are three of the five you saw at the end of session 2, in the same order and with the same
names:

    vorhersage = modell(Xtr)        the forward pass
    fehler = verlust(...)           how wrong it is
    optimierer.zero_grad()          throw away the previous step's gradients
    fehler.backward()               work out this step's
    optimierer.step()               move the weights

The last two are already in the cell. Write the first three above them.

If `zero_grad()` is missing, PyTorch adds the new gradients to the old ones instead of replacing
them, and the model walks off in a direction that is the sum of everything it has ever wanted to
do. Worth causing once on purpose, after the cell runs.

If you are stuck for ten minutes, the solution is released, see `kursregeln.md`. Nobody loses
the actual task over a bracket.

In [ ]:
def als_tensor(werte, mitte, spanne):
    return torch.tensor((werte - mitte) / spanne, dtype=torch.float32).unsqueeze(1)


Xtr, Ytr = als_tensor(k[i_tr], k_m, k_s), als_tensor(y[i_tr], y_m, y_s)
Xva, Yva = als_tensor(k[i_va], k_m, k_s), als_tensor(y[i_va], y_m, y_s)
Xte, Yte = als_tensor(k[i_te], k_m, k_s), als_tensor(y[i_te], y_m, y_s)


def baue(neuronen=64, samen=0):
    torch.manual_seed(samen)
    return nn.Sequential(nn.Linear(1, neuronen), nn.ReLU(),
                         nn.Linear(neuronen, neuronen), nn.ReLU(),
                         nn.Linear(neuronen, 1))


def trainiere(modell, schritte=6000, lernrate=0.01, alle=25):
    """Returns the two curves and a copy of the weights at every measured point.

    `stellen` counts completed updates, so the first entry is 1, not 0, and the last
    entry is always the final state of the model.
    """
    verlust = nn.MSELoss()
    optimierer = torch.optim.Adam(modell.parameters(), lr=lernrate)
    stellen, kurve_tr, kurve_va, zustaende = [], [], [], []
    for schritt in range(schritte):

        vorhersage = modell(Xtr)
        fehler = verlust(vorhersage, Ytr)
        optimierer.zero_grad()
        fehler.backward()
        optimierer.step()

        getan = schritt + 1
        if getan % alle == 0 or getan == schritte:
            with torch.no_grad():
                stellen.append(getan)
                kurve_tr.append(float(verlust(modell(Xtr), Ytr)))
                kurve_va.append(float(verlust(modell(Xva), Yva)))
                zustaende.append({n: t.detach().clone()
                                  for n, t in modell.state_dict().items()})
    return np.array(stellen), np.array(kurve_tr), np.array(kurve_va), zustaende


laeufe = {}
for breite in (2, 4, 16, 64):
    m = baue(breite)
    st, ktr, kva, zust = trainiere(m)
    laeufe[breite] = dict(stellen=st, tr=ktr, va=kva, zustaende=zust,
                          tief=int(np.argmin(kva)))

lauf = laeufe[64]
tief = lauf["tief"]
print(f"64 neurons: lowest validation error {lauf['va'][tief]:.3f} "
      f"after {lauf['stellen'][tief]} updates")
print(f"            after all {lauf['stellen'][-1]} updates: {lauf['va'][-1]:.3f}")
print(f"            training error at the end: {lauf['tr'][-1]:.3f}")

Two curves again, now over training steps instead of over degrees. The training error falls
overall, though not on every single interval: Adam is not least squares, and a step that helps on
average can make things briefly worse. The validation error falls, bottoms out, and rises. The
dashed line marks the lowest point.

Say it carefully: that point estimates when you should have stopped, on this validation set. It
does not mark a moment at which learning turned into memorising. The two were mixed from the
first update; the mixture simply got worse.

Notice also how much gentler the rise is than the polynomial's. In this run the validation error
climbs by about a third from its minimum, where the polynomial's validation error climbed by a
factor of forty inside the training range, from degree six to twelve, and by about eight
orders of magnitude outside it. One run does not prove
a general law, but it is worth knowing
that the textbook picture of a violently rising validation curve is drawn from models with a
sharper capacity dial than a ReLU network trained with Adam.

In [ ]:
stellen, kurve_tr, kurve_va = lauf["stellen"], lauf["tr"], lauf["va"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=stellen, y=kurve_tr, mode="lines", name="training",
                         line=dict(width=3, color="#4c78a8")))
fig.add_trace(go.Scatter(x=stellen, y=kurve_va, mode="lines", name="validation",
                         line=dict(width=3, color="crimson")))
fig.add_vline(x=stellen[tief], line_dash="dash", line_color="#d8a24a")
fig.add_annotation(x=np.log10(stellen[tief]), y=np.log10(kurve_va.max()),
                   text="stop here", showarrow=False, xshift=45,
                   font=dict(color="#d8a24a"))
fig.update_layout(height=470, xaxis_title="updates", yaxis_title="error, in scaled units",
                  yaxis_type="log", xaxis_type="log",
                  title="64 neurons per layer, 25 buildings to learn from")
fig.show()

The slider walks through the same run. Left the two curves up to that point, right what the model
looks like on the data.

In [ ]:
gitter = torch.tensor((gitter_k - k_m) / k_s, dtype=torch.float32).unsqueeze(1)
proben = sorted(set(list(range(0, len(stellen), max(1, len(stellen) // 25)))
                    + [len(stellen) - 1]))

hilfs = baue()
linien = []
for j in proben:
    hilfs.load_state_dict(lauf["zustaende"][j])
    with torch.no_grad():
        linien.append(hilfs(gitter).numpy().ravel() * y_s + y_m)

fig = make_subplots(rows=1, cols=2, subplot_titles=("error so far", "the model on the data"))
fig.add_trace(go.Scatter(x=stellen[:1], y=kurve_tr[:1], mode="lines+markers",
                         line=dict(width=3, color="#4c78a8")), row=1, col=1)
fig.add_trace(go.Scatter(x=stellen[:1], y=kurve_va[:1], mode="lines+markers",
                         line=dict(width=3, color="crimson")), row=1, col=1)
fig.add_trace(go.Scatter(x=k[i_va], y=y[i_va], mode="markers",
                         marker=dict(size=4, color="#8b95a8", opacity=0.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=k[i_tr], y=y[i_tr], mode="markers",
                         marker=dict(size=8, color="crimson")), row=1, col=2)
fig.add_trace(go.Scatter(x=gitter_k, y=linien[0], mode="lines",
                         line=dict(width=3, color="#d8a24a")), row=1, col=2)

fig.frames = [go.Frame(name=str(stellen[j]),
                       data=[go.Scatter(x=stellen[:j + 1], y=kurve_tr[:j + 1],
                                        mode="lines+markers",
                                        line=dict(width=3, color="#4c78a8")),
                             go.Scatter(x=stellen[:j + 1], y=kurve_va[:j + 1],
                                        mode="lines+markers",
                                        line=dict(width=3, color="crimson")),
                             go.Scatter(x=gitter_k, y=linien[n], mode="lines",
                                        line=dict(width=3, color="#d8a24a"))],
                       traces=[0, 1, 4])
              for n, j in enumerate(proben)]

fig.update_layout(height=520, showlegend=False,
                  sliders=[dict(active=0, currentvalue=dict(prefix="updates: "),
                                steps=[dict(method="animate", label=str(stellen[j]),
                                            args=[[str(stellen[j])],
                                                  dict(mode="immediate",
                                                       frame=dict(duration=0, redraw=False))])
                                       for j in proben])])
fig.update_xaxes(title_text="updates", type="log",
                 range=[0, np.log10(stellen[-1])], row=1, col=1)
fig.update_yaxes(title_text="error", type="log",
                 range=[np.log10(min(kurve_tr.min(), kurve_va.min()) * 0.8),
                        np.log10(max(kurve_tr.max(), kurve_va.max()) * 1.2)], row=1, col=1)
fig.update_xaxes(title_text="compactness",
                 range=[k[sichtbar].min() - 0.2, k[sichtbar].max() + 0.2], row=1, col=2)
fig.update_yaxes(title_text="target",
                 range=[y[sichtbar].min() - 20, y[sichtbar].max() + 20], row=1, col=2)
fig.show()

## Two ways out, and how much each is worth

The first is to use less model. That worked spectacularly on the polynomial: degree five beat
degree nine by a factor of two inside the training range, and by six orders of magnitude outside
it. The four widths were already trained above, so here they are, each shown at its own best
point.

In [ ]:
breiten = [2, 4, 16, 64]
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"{b} neurons per layer" for b in breiten])
for n, breite in enumerate(breiten):
    e = laeufe[breite]
    hilfs = baue(breite)
    hilfs.load_state_dict(e["zustaende"][e["tief"]])
    with torch.no_grad():
        linie = hilfs(gitter).numpy().ravel() * y_s + y_m
    r, c = divmod(n, 2)
    fig.add_trace(go.Scatter(x=k[i_tr], y=y[i_tr], mode="markers", showlegend=False,
                             marker=dict(size=7, color="crimson")), row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=gitter_k, y=linie, mode="lines", showlegend=False,
                             line=dict(width=3, color="#d8a24a")), row=r + 1, col=c + 1)
fig.update_layout(height=640, title="four widths, each stopped where its validation error was "
                                    "lowest")
fig.update_yaxes(range=[y[sichtbar].min() - 20, y[sichtbar].max() + 20])
fig.update_xaxes(range=[k[sichtbar].min() - 0.2, k[sichtbar].max() + 0.2])
fig.show()

print(f"{'width':>6} {'best':>8} {'at update':>11} {'at the end':>12} {'saved by stopping':>19}")
for breite in breiten:
    e = laeufe[breite]
    j = e["tief"]
    gewinn = 100 * (e["va"][-1] - e["va"][j]) / e["va"][-1]
    print(f"{breite:>6} {e['va'][j]:>8.3f} {e['stellen'][j]:>11} {e['va'][-1]:>12.3f} "
          f"{gewinn:>18.1f}%")

Read the numbers before you conclude anything, because they are not what the textbook picture
promises. In this run, four, sixteen and sixty-four neurons land within about five percent of
each other, and the differences between them are small enough that a different random split
could reorder them. Only two neurons is clearly behind, and here the problem looks like the
opposite one: its curve stays too stiff to follow both turns. With two layers it could draw
them in principle; in this run it did not.

One split and one starting point cannot settle whether width matters in general. What they do
settle is that you cannot tell by looking at the architecture, which is the point: the validation
set decides, not the recipe.

The second way out is to stop at the lowest point of the validation curve instead of at the end.
That is what the copies of the weights in `trainiere` are for, and taking the one from the best
point is called *early stopping*. Here it is worth between two and twenty-two percent depending
on the width, for no extra training, though note that this implementation still runs every step
and keeps every checkpoint, so it saves memory and time not at all.

And be honest about what that number is: we picked the best of 240 checkpoints using the
validation set, so an improvement over the last checkpoint was close to guaranteed. Whether it
also helps on genuinely new data is a question the validation set cannot answer. That is what the
test set is for, and we get exactly one look at it.

## Opening the test set

Once, now, together. The decision rule is stated before the number is read: take the width with
the lowest validation error, at its own best checkpoint, and apply it to the hundred and sixty
buildings nobody has touched.

<img src="img/wachssiegel.jpg" width="230">

*Photo: postal authorities of the Russian Empire, 1855,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Reval_1855-02-09_church_letter_wax_seal_(cropped).jpg),
public domain.*

What the number means: an estimate of how this model does on new buildings drawn the same way.
What it does not mean: anything about another district or another construction period. A few
test buildings lie outside the range we trained on, so the number contains some extrapolation,
but it does not show how the model does on those alone, and nothing about compactness beyond what
was tested. You saw two such buildings break a polynomial earlier.

> You get one look at the test set. Tune anything after looking at it and you no longer have one.

In [ ]:
gewaehlt = min(breiten, key=lambda b: laeufe[b]["va"][laeufe[b]["tief"]])
e = laeufe[gewaehlt]
bestes = baue(gewaehlt)
bestes.load_state_dict(e["zustaende"][e["tief"]])

with torch.no_grad():
    fehler_va = float(nn.MSELoss()(bestes(Xva), Yva))
    fehler_te = float(nn.MSELoss()(bestes(Xte), Yte))

print(f"chosen by validation: {gewaehlt} neurons, stopped after {e['stellen'][e['tief']]} updates")
for name, idx in (("training", i_tr), ("validation", i_va), ("test", i_te)):
    print(f"compactness in the {name} buildings: {k[idx].min():.2f} to {k[idx].max():.2f}")
print(f"validation: {fehler_va:.3f} scaled, typical miss {np.sqrt(fehler_va) * y_s:.1f} kWh/m2a")
print(f"test      : {fehler_te:.3f} scaled, typical miss {np.sqrt(fehler_te) * y_s:.1f} kWh/m2a")

# what the exact generating rule scores on the same buildings, noise and all
for name, idx in (("validation", i_va), ("test", i_te)):
    print(f"the rule itself on {name}: "
          f"{np.sqrt(np.mean((rein[idx] - y[idx]) ** 2)):.1f} kWh/m2a")

The typical miss lands near the twenty we put in, and the last two lines say why that is the
right comparison. Even the exact rule that generated the data, handed to you for free, misses by
about nineteen on these buildings, because the noise is still there.

Twenty is the expected error on fresh observations, not a floor any particular sample has to
respect: a lucky eighty buildings can score below it, and ours nearly do. What it does mean is
that a model reporting three on its training data is telling you about its training data.

## Tests

This cell has to run green before you submit. It checks three things: that the loop moves the
weights, that the gradients are cleared between updates, and that the loop trains on the training
set and not on anything else.

In [ ]:
# 1. the weights move at all
probe = baue(16, samen=5)
vorher = [p.detach().clone() for p in probe.parameters()]
st_, ktr_, kva_, zust_ = trainiere(probe, schritte=600)
assert any(not torch.allclose(a, b) for a, b in zip(vorher, probe.parameters())), (
    "the weights did not move; is optimierer.step() in the loop?")
assert np.isfinite(ktr_).all() and np.isfinite(kva_).all(), "the error became inf or nan"
assert ktr_[-1] < ktr_[0], "the training error has to fall"

# 2. gradients are cleared: after two updates, the stored gradient must be the gradient of
#    the second forward pass alone, not the sum of both
zwei = baue(16, samen=5)
st2, _, _, zust2 = trainiere(zwei, schritte=2, alle=1)
kontrolle = baue(16, samen=5)
kontrolle.load_state_dict(zust2[0])                 # the weights after the first update
nn.MSELoss()(kontrolle(Xtr), Ytr).backward()
for p, q in zip(zwei.parameters(), kontrolle.parameters()):
    assert p.grad is not None, "no gradients at all; is fehler.backward() reached?"
    assert torch.allclose(p.grad, q.grad, atol=1e-5), (
        "the gradients of two updates were added together; zero_grad() is missing or misplaced")

# 3. the loop does not learn from the validation targets
Yva_echt = Yva.clone()
a = baue(16, samen=9)
trainiere(a, schritte=120)
Yva += 500.0
b = baue(16, samen=9)
trainiere(b, schritte=120)
Yva.copy_(Yva_echt)
for p, q in zip(a.parameters(), b.parameters()):
    assert torch.allclose(p, q), "changing the validation targets changed the trained weights"

gerade = np.polyfit(k[i_tr], y[i_tr], 1)
rest_gerade = float(np.mean((np.polyval(gerade, k[i_va]) - y[i_va]) ** 2) / y_s ** 2)
assert kva_.min() < rest_gerade, (
    f"a straight line does better than your network: {kva_.min():.3f} against {rest_gerade:.3f}")

print(f"all green: best validation {kva_.min():.3f}, straight line {rest_gerade:.3f}")

## Where you stand now

You can write a training loop in PyTorch and say what each line does. You can split a dataset
into three parts in the right order and explain why the scaling comes out of one of them. You can
read a pair of error curves and name the point at which you should have stopped, and you know
that the point was chosen using the same data it is being judged on. You have seen the same
remedy work spectacularly on one model and barely at all on another, which should cure you of
applying recipes without measuring. And you can tell overfitting from extrapolation, which are
two different failures with very different sizes.

Next session the input stops being a number somebody chose for you. It becomes the floor plan.

---

### Terms to learn

| term | in one sentence |
|---|---|
| overfitting | validation error rises again while training error keeps falling |
| extrapolation | predicting outside the range the model was fitted on; usually much worse |
| capacity | how much a model can represent; degree, width and depth feed it |
| training set | what the model learns from |
| validation set | what you look at while deciding anything about the model |
| test set | opened once, at the end; an estimate for new data of the same kind |
| leakage | information from held-out data reaching the model, usually through scaling |
| early stopping | keeping the weights from the best validation checkpoint instead of the last |
| regularisation | anything that trades training accuracy for behaviour on new data |
| expected error on fresh data | what the noise level predicts; a single sample can beat it by luck |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

Set the network up on the noisy data so that the validation error is as small as you can get it
within a fixed experiment, freeze it at its best point, and submit it.

1. Choose two learning rates between 0.001 and 0.05 and a seed of your own. The first cell
   trains three widths (4, 16 and 64 neurons per layer) at each of your rates, six runs of 6000
   updates, about twenty seconds. You do not have to search further.
2. The second cell takes the run with the lowest validation error, shows its curves, and saves
   it as `abgabe_03.pt`. Submit that file with the notebook.

The training and validation buildings stay as they are. The test buildings stay shut: when you
rerun the whole notebook, the class cell that opened them prints its number again; that is the
one look from class, and nothing you decide may depend on it.

Your submitted model is graded on a further set of houses that nobody has seen, drawn the same
way. A particularly good number there earns no point; what earns a point is that you predicted it
sensibly. A flat validation curve is a legitimate observation and costs nothing, as long as you
show it and read it correctly.

In [ ]:
breiten = [4, 16, 64]

# YOUR CHOICE: two learning rates between 0.001 and 0.05, and a whole number as your seed
lernraten = [None, None]
mein_samen = None

assert len(lernraten) == 2 and None not in lernraten, "enter exactly two learning rates"
assert mein_samen is not None, "enter a seed"
assert all(0.001 <= lr <= 0.05 for lr in lernraten), "both rates between 0.001 and 0.05"
assert lernraten[0] != lernraten[1], "two different rates"
assert isinstance(mein_samen, int), "the seed is a whole number"

protokoll = []
for lr in lernraten:
    for n in breiten:
        m = baue(n, samen=mein_samen)
        st, ktr, kva, zust = trainiere(m, schritte=6000, lernrate=lr)
        if not np.isfinite(kva).all():
            print(f"{n:4d} neurons, learning rate {lr:<6} ran away (inf or nan), left out")
            continue
        j = int(np.argmin(kva))
        protokoll.append(dict(neuronen=n, lernrate=lr, stelle=int(st[j]), val=float(kva[j]),
                              zustand=zust[j], stellen=st, tr=ktr, va=kva))
        print(f"{n:4d} neurons, learning rate {lr:<6} best validation {kva[j]:.4f} "
              f"after {st[j]:5d} updates, at the end {kva[-1]:.3f}")
    print()

# what the scaling would have been, had it come from all 265 buildings
print(f"scaling from training only: mean {k_m:.3f}, spread {k_s:.3f}")
print(f"scaling from all 265      : mean {k.mean():.3f}, spread {k.std():.3f}")

In [ ]:
assert protokoll, "every run ran away; choose smaller learning rates"
wahl = min(protokoll, key=lambda e: e["val"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=wahl["stellen"], y=wahl["tr"], mode="lines", name="training",
                         line=dict(width=3, color="#4c78a8")))
fig.add_trace(go.Scatter(x=wahl["stellen"], y=wahl["va"], mode="lines", name="validation",
                         line=dict(width=3, color="crimson")))
fig.add_vline(x=wahl["stelle"], line_dash="dash", line_color="#d8a24a")
fig.update_layout(height=430, xaxis_type="log", yaxis_type="log", xaxis_title="updates",
                  yaxis_title="error, in scaled units",
                  title=f"your choice: {wahl['neuronen']} neurons, learning rate "
                        f"{wahl['lernrate']}, frozen after {wahl['stelle']} updates")
fig.show()

print(f"validation error {wahl['val']:.3f} scaled, "
      f"typical miss {np.sqrt(wahl['val']) * y_s:.1f} kWh/m2a")
print(f"the rule itself, noise and all, on the validation buildings: "
      f"{np.sqrt(np.mean((rein[i_va] - y[i_va]) ** 2)):.1f} kWh/m2a")
print(f"training range of compactness: {k_lo:.2f} to {k_hi:.2f}")

torch.save(dict(neuronen=wahl["neuronen"], zustand=wahl["zustand"],
                k_m=float(k_m), k_s=float(k_s), y_m=float(y_m), y_s=float(y_s)),
           "abgabe_03.pt")
print("written: abgabe_03.pt, submit it together with this notebook")

### Your interpretation

1. **Where you stopped.** Give the point at which your chosen run was frozen, and the reason, read off its curve. If the curve is flat, say so and say what that means.
2. **Network size.** Compare the three widths at the same learning rate, for each of your two rates, with the numbers. Is the order from best to worst the same at both rates, or do any two widths swap places? Widths within 0.002 of each other count as a tie. Say how much of the difference you would trust with one split and one seed.
3. **Split, then scale.** The first cell prints the scaling from the training buildings and the scaling had it come from all 265. Say what would have leaked, and why your validation error would then no longer be an independent check.
4. **The test number.** Write down the typical miss in kWh/m²a you expect on the unseen houses, with your reason. Then say what that number will measure once it is computed.
5. **One limit.** Name one kind of building or situation for which that number says nothing, and why the evaluation does not cover it.
6. **Design consequence.** A designer wants to use your model to compare two variants of the same building. Say what your model could and could not tell them. Refusing, with a reason, is a valid answer.

*Your text here.*